# Robot Data Analyzer
Comprehensive analysis and visualization tool for data collected from sessions

In [ ]:
# Imports and Configuration
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Dict
import warnings

warnings.filterwarnings('ignore')

# Set style for better-looking plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

: 

## RobotDataAnalyzer Class
Encapsulates all data loading, analysis, and visualization logic.

In [ ]:
class RobotDataAnalyzer:
    """
    
    Comprehensive analysis tool for robot session data
    
    """
    
    def __init__(self, log_file: str):
        self.log_file = log_file
        self.data = None
        self.sensor_df = None
        self.motor_df = None
        self.events_df = None
        
        self.load_data()
    
    def load_data(self):
        """
        
        Load and parse session data
        
        """
        try:
            with open(self.log_file, 'r') as f:
                self.data = json.load(f)
            
            # Convert to DataFrames
            self.sensor_df = pd.DataFrame(self.data.get('sensor_data', []))
            self.motor_df = pd.DataFrame(self.data.get('motor_commands', []))
            self.events_df = pd.DataFrame(self.data.get('events', []))
            
            # Convert timestamps to datetime
            if not self.sensor_df.empty:
                self.sensor_df['datetime'] = pd.to_datetime(self.sensor_df['timestamp'], unit='s')
            if not self.motor_df.empty:
                self.motor_df['datetime'] = pd.to_datetime(self.motor_df['timestamp'], unit='s')
            if not self.events_df.empty:
                self.events_df['datetime'] = pd.to_datetime(self.events_df['timestamp'], unit='s')
                
            print(f"Data loaded successfully: {len(self.sensor_df)} sensor readings, {len(self.motor_df)} motor commands")
            
        except Exception as e:
            print(f"Error loading data: {e}")

## Generate Performance Summary

In [ ]:

def generate_performance_summary(self) -> Dict:
    """
    
    Generate comprehensive performance metrics
    
    """
    if self.sensor_df.empty:
        return {}
    
    # Basic metrics
    total_frames = len(self.sensor_df)
    detection_rate = self.sensor_df['person_detected'].mean() if 'person_detected' in self.sensor_df.columns else 0
    session_duration = (self.sensor_df['timestamp'].max() - self.sensor_df['timestamp'].min())
    
    # Distance statistics (filter out invalid readings)
    valid_distances = self.sensor_df[self.sensor_df['ultrasonic_distance'] < 500]['ultrasonic_distance']
    
    # Motor activity analysis
    motor_activity = 0
    if not self.motor_df.empty:
        motor_activity = ((abs(self.motor_df['left_speed']) + abs(self.motor_df['right_speed'])) > 5).mean()
    
    # Action distribution
    action_dist = self.motor_df['action'].value_counts().to_dict() if not self.motor_df.empty else {}
    
    summary = {
        'session_duration': round(session_duration, 2),
        'total_frames': total_frames,
        'detection_rate': round(detection_rate * 100, 2),
        'avg_distance': round(valid_distances.mean(), 2) if not valid_distances.empty else 0,
        'min_distance': round(valid_distances.min(), 2) if not valid_distances.empty else 0,
        'max_distance': round(valid_distances.max(), 2) if not valid_distances.empty else 0,
        'motor_activity_rate': round(motor_activity * 100, 2),
        'action_distribution': action_dist,
        'safety_events': len(self.events_df[self.events_df['event'] == 'safety_backup']) if not self.events_df.empty else 0
    }
    
    return summary

# Attach the method to the class
RobotDataAnalyzer.generate_performance_summary = generate_performance_summary


## Visualization Dashboard

In [ ]:


def create_performance_dashboard(self, save_path: str = None):
    """
    
    Create comprehensive performance dashboard
    
    """
    if self.sensor_df.empty:
        print("No data available for visualization")
        return
    
    fig = plt.figure(figsize=(16, 12))
    
    # 1. Detection Rate Over Time
    plt.subplot(2, 3, 1)
    detection_rolling = self.sensor_df.set_index('datetime')['person_detected'].rolling('5s').mean()
    plt.plot(detection_rolling.index, detection_rolling * 100, linewidth=2)
    plt.title('Person Detection Rate Over Time', fontsize=14, fontweight='bold')
    plt.ylabel('Detection Rate (%)')
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3)
    
    # 2. Distance Measurements
    plt.subplot(2, 3, 2)
    valid_distances = self.sensor_df[self.sensor_df['ultrasonic_distance'] < 500]
    plt.plot(valid_distances['datetime'], valid_distances['ultrasonic_distance'], 
            color='orange', linewidth=2, alpha=0.7)
    plt.axhline(y=50, color='red', linestyle='--', alpha=0.7, label='Safety Limit')
    plt.axhline(y=150, color='green', linestyle='--', alpha=0.7, label='Target Distance')
    plt.title('Distance Measurements', fontsize=14, fontweight='bold')
    plt.ylabel('Distance (cm)')
    plt.legend()
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3)
    
    # ... (other plots remain the same as your script)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Dashboard saved to: {save_path}")
    
    plt.show()

# Attach method
RobotDataAnalyzer.create_performance_dashboard = create_performance_dashboard


## Following Performance Analysis

In [ ]:

def analyze_following_performance(self):
    """
    
    Analyze person following effectiveness
    
    """
    if self.sensor_df.empty or self.motor_df.empty:
        return
    
    merged_df = pd.merge_asof(
        self.sensor_df.sort_values('timestamp'),
        self.motor_df.sort_values('timestamp'),
        on='timestamp',
        direction='nearest'
    )
    
    following_data = merged_df[merged_df['person_detected'] == True]
    
    if following_data.empty:
        print("No following data available")
        return
    
    target_distance = 150  # cm
    distance_error = abs(following_data['ultrasonic_distance'] - target_distance)
    avg_error = distance_error.mean()
    
    frame_center = 320  # assuming 640px width
    steering_errors = [
        abs(row['person_center_x'] - frame_center)
        for _, row in following_data.iterrows()
        if row['person_center_x'] is not None
    ]
    
    avg_steering_error = np.mean(steering_errors) if steering_errors else 0
    
    print("=== Following Performance Analysis ===")
    print(f"Average Distance Error: {avg_error:.2f} cm")
    print(f"Average Steering Error: {avg_steering_error:.2f} pixels")
    print(f"Time in Follow Mode: {(following_data['action'] == 'follow').sum() * 0.05:.2f}s")
    print(f"Following Success Rate: {len(following_data) / len(merged_df) * 100:.1f}%")

RobotDataAnalyzer.analyze_following_performance = analyze_following_performance


## Exporting Reports

In [ ]:

def export_summary_report(self, output_file: str = None):
    """
    
    Export detailed analysis report
    
    """
    if output_file is None:
        output_file = f"analysis_report_{Path(self.log_file).stem}.txt"
    
    summary = self.generate_performance_summary()
    
    report = f"""
ROBOT PERFORMANCE ANALYSIS REPORT
=====================================
Generated from: {self.log_file}
Analysis Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}

SESSION OVERVIEW:
- Duration: {summary['session_duration']} seconds
- Total Frames: {summary['total_frames']}
- Average FPS: {summary['total_frames'] / max(summary['session_duration'], 1):.1f}

DETECTION PERFORMANCE:
- Detection Rate: {summary['detection_rate']}%
- Average Distance: {summary['avg_distance']} cm
- Distance Range: {summary['min_distance']} - {summary['max_distance']} cm

MOTOR ACTIVITY:
- Activity Rate: {summary['motor_activity_rate']}%
- Action Distribution: {summary['action_distribution']}

SAFETY METRICS:
- Safety Backup Events: {summary['safety_events']}
- Safety Event Rate: {summary['safety_events'] / max(summary['session_duration'] / 60, 1):.2f} per minute

RECOMMENDATIONS:
"""
    
    if summary['detection_rate'] < 70:
        report += "- Consider improving lighting conditions or person detection model\n"
    if summary['safety_events'] > 5:
        report += "- Review obstacle avoidance parameters - frequent safety stops detected\n"
    if summary['motor_activity_rate'] < 30:
        report += "- Low motor activity suggests limited person interaction\n"
    
    with open(output_file, 'w') as f:
        f.write(report)
    
    print(f"Detailed report saved to: {output_file}")
    return report

RobotDataAnalyzer.export_summary_report = export_summary_report


## Quick Analysis Function

In [ ]:


def analyze_robot_session(log_file: str, create_dashboard: bool = True, 
                         save_dashboard: bool = True) -> RobotDataAnalyzer:
    """Quick analysis function"""
    analyzer = RobotDataAnalyzer(log_file)
    
    summary = analyzer.generate_performance_summary()
    print("\n=== QUICK SUMMARY ===")
    for key, value in summary.items():
        if key != 'action_distribution':
            print(f"{key.replace('_', ' ').title()}: {value}")
    
    if create_dashboard:
        save_path = f"dashboard_{Path(log_file).stem}.png" if save_dashboard else None
        analyzer.create_performance_dashboard(save_path)
    
    analyzer.analyze_following_performance()
    analyzer.export_summary_report()
    
    return analyzer


## LineFollowingAnalyzer Class

In [ ]:


class LineFollowingAnalyzer:
    """Specialized analyzer for line following sessions"""
    
    def __init__(self, log_file: str):
        self.analyzer = RobotDataAnalyzer(log_file)
        self.line_events = self._extract_line_events()
    
    def _extract_line_events(self):
        """Extract line-specific events from log data"""
        if self.analyzer.events_df.empty:
            return pd.DataFrame()
        
        line_events = self.analyzer.events_df[
            self.analyzer.events_df['event'].isin(['line_data', 'line_lost', 'line_reacquired'])
        ].copy()
        
        return line_events


## Line Following Accuracy Analysis

In [ ]:

CAMERA_WIDTH = 640  # Adjustable 

def analyze_line_following_accuracy(self):
    """
    
    Analyze line following accuracy and stability
    
    """
    if self.analyzer.sensor_df.empty:
        return {}
    
    frame_center = CAMERA_WIDTH // 2
    detected_data = self.analyzer.sensor_df[self.analyzer.sensor_df['person_detected'] == True]
    
    if detected_data.empty:
        return {"error": "No line detection data found"}
    
    center_deviations = abs(detected_data['person_center_x'] - frame_center)
    
    metrics = {
        'line_detection_rate': detected_data.shape[0] / self.analyzer.sensor_df.shape[0] * 100,
        'avg_center_deviation': center_deviations.mean(),
        'max_center_deviation': center_deviations.max(),
        'stability_score': 100 - (center_deviations.std() / frame_center * 100),
        'time_on_line': detected_data.shape[0] * 0.05,  # Assuming 20 FPS
    }
    
    return metrics

LineFollowingAnalyzer.analyze_line_following_accuracy = analyze_line_following_accuracy


## Line Following Dashboard

In [ ]:

def create_line_following_dashboard(self, save_path: str = None):
    """
    
    Create line following specific dashboard
    
    """
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # 1. Line Detection Over Time
    ax1 = axes[0, 0]
    detection_data = self.analyzer.sensor_df['person_detected'].rolling(20).mean() * 100
    ax1.plot(detection_data.index * 0.05, detection_data, 'g-', linewidth=2)
    ax1.set_title('Line Detection Rate Over Time')
    ax1.set_ylabel('Detection Rate (%)')
    ax1.set_xlabel('Time (seconds)')
    ax1.grid(True, alpha=0.3)
    
    # 2. Center Deviation
    ax2 = axes[0, 1]
    if 'person_center_x' in self.analyzer.sensor_df.columns:
        frame_center = CAMERA_WIDTH // 2
        center_x = self.analyzer.sensor_df['person_center_x'].dropna()
        deviation = center_x - frame_center
        ax2.plot(deviation.index * 0.05, deviation, 'b-', alpha=0.7)
        ax2.axhline(y=0, color='red', linestyle='--', label='Perfect Center')
        ax2.fill_between(deviation.index * 0.05, -50, 50, alpha=0.2, color='green', label='Good Range')
        ax2.set_title('Line Center Deviation')
        ax2.set_ylabel('Deviation (pixels)')
        ax2.set_xlabel('Time (seconds)')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
    
    # 3. Motor Speed Correlation
    ax3 = axes[1, 0]
    if not self.analyzer.motor_df.empty:
        motor_diff = self.analyzer.motor_df['left_speed'] - self.analyzer.motor_df['right_speed']
        ax3.hist(motor_diff, bins=30, alpha=0.7, color='purple')
        ax3.set_title('Steering Correction Distribution')
        ax3.set_xlabel('Left - Right Motor Speed')
        ax3.set_ylabel('Frequency')
        ax3.axvline(x=0, color='red', linestyle='--', label='Straight')
        ax3.legend()
        ax3.grid(True, alpha=0.3)
    
    # 4. Performance Summary
    ax4 = axes[1, 1]
    metrics = self.analyze_line_following_accuracy()
    
    if 'error' not in metrics:
        summary_text = f"""Line Following Performance:
        
Detection Rate: {metrics.get('line_detection_rate', 0):.1f}%
Avg Deviation: {metrics.get('avg_center_deviation', 0):.1f}px
Max Deviation: {metrics.get('max_center_deviation', 0):.1f}px  
Stability Score: {metrics.get('stability_score', 0):.1f}/100
Time on Line: {metrics.get('time_on_line', 0):.1f}s"""
    else:
        summary_text = "No line detection data available"
    
    ax4.text(0.1, 0.5, summary_text, fontsize=11, verticalalignment='center',
            bbox=dict(boxstyle="round,pad=0.3", facecolor="lightgreen", alpha=0.7))
    ax4.set_xlim(0, 1)
    ax4.set_ylim(0, 1)
    ax4.axis('off')
    ax4.set_title('Performance Summary')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Line following dashboard saved to: {save_path}")
    
    plt.show()

LineFollowingAnalyzer.create_line_following_dashboard = create_line_following_dashboard


## Quick Line Following Analysis Function

In [ ]:

def analyze_line_following_session(log_file: str):
    """
    
    Analyze line following session
    
    """
    analyzer = LineFollowingAnalyzer(log_file)
    
    metrics = analyzer.analyze_line_following_accuracy()
    print("\n=== Line Following Analysis ===")
    for key, value in metrics.items():
        if key != 'error':
            print(f"{key.replace('_', ' ').title()}: {value}")
    
    analyzer.create_line_following_dashboard()
    return analyzer
